# 00 - Environment Setup & Model Downloads

## Overview
- Download all required pretrained models to local `../models/` directory
- Install dependencies (isc_feature_extractor, etc.)
- Verify all paths resolve correctly
- **No training required** — all models used as-is from pretrained weights

## 1. Install Dependencies

In [1]:
# Install ISC21 feature extractor (official pretrained ISC model)
!pip install git+https://github.com/lyakaap/ISC21-Descriptor-Track-1st --quiet

## 2. Paths & Environment

In [1]:
import os, sys, urllib.request, time
from pathlib import Path

# Base paths
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

# ── Model files to verify/download ──
REQUIRED_MODELS = {
    'MAST3R': {
        'file': MODEL_DIR / 'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth',
        'url': 'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth',
        'min_size_mb': 100,
    },
    'DINOv2': {
        'file': MODEL_DIR / 'dinov2-base' / 'model.safetensors',
        'url': None,  # Manual download from HuggingFace: facebook/dinov2-base
        'min_size_mb': 0,
    },
}

print('Model directory:', MODEL_DIR.resolve())
os.environ['HF_HOME'] = str(MODEL_DIR.resolve())

Model directory: /root/autodl-tmp/Image-Matching-and-3D-Reconstruction/models


## 3. Download MAST3R Checkpoint

In [2]:
def download_with_resume(url, dest_path, chunk_size=8*1024*1024):
    """Download with resume support. Handles servers that don't provide Content-Length."""
    dest_path = Path(dest_path).resolve()
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Get remote size (may be 0 if server doesn't support HEAD)
    total_size = 0
    try:
        req = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(req, timeout=30) as resp:
            total_size = int(resp.headers.get('Content-Length', 0))
    except Exception:
        pass  # HEAD not supported, will download without progress
    
    size_str = f'{total_size / 1024**3:.1f} GB' if total_size > 0 else 'unknown size'
    print(f'Remote size: {size_str}')
    
    downloaded = dest_path.stat().st_size if dest_path.exists() else 0
    # Only skip if we know the total size AND the file matches
    if total_size > 0 and downloaded >= total_size:
        print(f'Already complete: {dest_path}')
        return
    if total_size == 0 and downloaded > 0:
        print(f'File exists ({downloaded/1024:.1f} KB), but cannot verify completeness (unknown remote size)')
    
    print(f'Downloading {"from " + str(downloaded) if downloaded else ""} ...')
    req = urllib.request.Request(url)
    if downloaded:
        req.add_header('Range', f'bytes={downloaded}-')
    
    mode = 'ab' if downloaded else 'wb'
    with urllib.request.urlopen(req, timeout=60) as resp:
        with open(dest_path, mode) as f:
            while True:
                chunk = resp.read(chunk_size)
                if not chunk: break
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    print(f'\r  {downloaded/total_size*100:.1f}% ({downloaded/1024**3:.2f} GB)', end='')
                else:
                    print(f'\r  {downloaded/1024**2:.1f} MB', end='')
    print(f'\nDone: {dest_path}')

# Download MAST3R
mast3r_info = REQUIRED_MODELS['MAST3R']
if not mast3r_info['file'].exists() or mast3r_info['file'].stat().st_size < mast3r_info['min_size_mb'] * 1024**2:
    download_with_resume(mast3r_info['url'], mast3r_info['file'])
else:
    print(f'MAST3R checkpoint OK: {mast3r_info["file"]}')

MAST3R checkpoint OK: ../models/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth


## 4. Verify DINOv2 (HuggingFace)

In [4]:
from transformers import AutoModel
dino_path = MODEL_DIR / 'dinov2-base'
if dino_path.exists():
    model = AutoModel.from_pretrained(str(dino_path), local_files_only=True)
    print(f'DINOv2-base loaded from {dino_path}')
    print(f'  Hidden size: {model.config.hidden_size}')
else:
    print(f'ERROR: {dino_path} not found!')
    print(f'  Download from: https://huggingface.co/facebook/dinov2-base')
    print(f'  Or run: huggingface-cli download facebook/dinov2-base --local-dir {dino_path}')


libgomp: Invalid value for environment variable OMP_NUM_THREADS


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2-base loaded from ../models/dinov2-base
  Hidden size: 768


## 5. Verify ISC21

In [7]:
from isc_feature_extractor import create_model
model, preprocessor = create_model(weight_name='isc_ft_v107', device='cpu')
print(f'ISC (isc_ft_v107) loaded. Output dim: 256')
del model  # Free memory

ISC (isc_ft_v107) loaded. Output dim: 256


## 6. Verify Checkpoints

In [8]:
# Verify trained ASMK/SPoC checkpoints exist
checkpoints = {
    'MAST3R-ASMK': '../checkpoints/mast3r_asmk/best_model.pth',
    'MAST3R-SPoC': '../checkpoints/mast3r_spoc/best_model.pth',
}

for name, path in checkpoints.items():
    if Path(path).exists():
        import torch
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        print(f'{name}: OK (epoch {ckpt.get("epoch", "?")}, loss {ckpt.get("loss", "?"):.4f})')
    else:
        print(f'{name}: MISSING ({path})')

MAST3R-ASMK: OK (epoch 1, loss 4.1445)
MAST3R-SPoC: OK (epoch 14, loss 0.0508)


## 6b. Download ALIKED Model (Keypoint Detector)

ALIKED-N16 weights for sparse keypoint extraction used in MASt3R hybrid matching.

In [9]:
# Download ALIKED-N16 model (used with LightGlue/MASt3R hybrid matching)
# Source: https://github.com/Shiaoming/ALIKED
ALIKED_DIR = MODEL_DIR / 'aliked'
ALIKED_DIR.mkdir(exist_ok=True)
ALIKED_WEIGHTS = ALIKED_DIR / 'aliked-n16.pth'

if not ALIKED_WEIGHTS.exists():
    print(f'Downloading ALIKED-N16 weights...')
    try:
        import torch.hub
        # Attempt download via torch hub
        model = torch.hub.load('Shiaoming/ALIKED', 'aliked-n16', pretrained=True, force_reload=True)
        torch.save(model.state_dict(), str(ALIKED_WEIGHTS))
        print(f'ALIKED-N16 saved to {ALIKED_WEIGHTS}')
    except Exception as e:
        print(f'torch.hub download failed: {e}')
        # Fallback: direct URL download
        try:
            url = 'https://github.com/Shiaoming/ALIKED/releases/download/v1.0/aliked-n16.pth'
            download_with_resume(url, str(ALIKED_WEIGHTS))
        except Exception as e2:
            print(f'Direct download also failed: {e2}')
            print('Please manually download ALIKED-N16 weights to:', ALIKED_WEIGHTS)
else:
    print(f'ALIKED-N16 OK: {ALIKED_WEIGHTS}')

ALIKED-N16 OK: ../models/aliked/aliked-n16.pth


## 6c. Download SuperPoint Model (Keypoint Detector)

MagicLeap SuperPoint weights for sparse keypoint extraction used in MASt3R hybrid matching.

In [10]:
# Download MagicLeap SuperPoint weights
# Champion used: keypoint_threshold=0.0005, max_keypoints=4096, nms_radius=4
SP_DIR = MODEL_DIR / 'superpoint'
SP_DIR.mkdir(exist_ok=True)
SP_WEIGHTS = SP_DIR / 'superpoint_v1.pth'

if not SP_WEIGHTS.exists():
    print(f'Downloading SuperPoint weights...')
    try:
        # MagicLeap SuperPoint weights (from hloc/megadepth training)
        url = 'https://github.com/magicleap/SuperGluePretrainedNetwork/raw/master/models/weights/superpoint_v1.pth'
        download_with_resume(url, str(SP_WEIGHTS))
    except Exception as e:
        print(f'Download failed: {e}')
        # Alternative: try kornia's SuperPoint
        print('Attempting alternative download via kornia...')
        try:
            import kornia
            # Kornia will cache the weights
            from kornia.feature import SuperPoint
            _ = SuperPoint(pretrained=True)
            # Find cached weights
            import glob
            cached = list(Path(torch.hub.get_dir()).rglob('superpoint*.pth'))
            if cached:
                import shutil
                shutil.copy(str(cached[0]), str(SP_WEIGHTS))
                print(f'SuperPoint cached and copied to {SP_WEIGHTS}')
        except Exception as e2:
            print(f'Alternative also failed: {e2}')
            print('Please manually download superpoint_v1.pth to:', SP_WEIGHTS)
else:
    print(f'SuperPoint OK: {SP_WEIGHTS}')

SuperPoint OK: ../models/superpoint/superpoint_v1.pth


## 6d. Verify curope CUDA Module (RoPE2D)

The champion notes that building curope with CUDA significantly speeds up MASt3R. Verify or document fallback.

In [11]:
# Verify curope (RoPE2D) CUDA module
# The champion built this with CUDA for speed. If missing, MASt3R falls back to PyTorch.
CUROPE_OK = False

# croco is NOT a pip package — it lives under mast3r/dust3r/croco/
# Add mast3r/dust3r/ to path so croco is importable
import sys
_DUST3R_PATH = Path('../mast3r/dust3r').resolve()
if str(_DUST3R_PATH) not in sys.path:
    sys.path.insert(0, str(_DUST3R_PATH))

try:
    from croco.models.curope.curope2d import cuRoPE2D
    # Test with a small tensor to verify CUDA availability
    import torch
    if torch.cuda.is_available():
        try:
            rope = cuRoPE2D(freq=100.0, F0=1.0)
            test_tokens = torch.randn(1, 4, 8, 64, device='cuda')
            test_pos = torch.zeros(1, 4, 2, dtype=torch.int64, device='cuda')
            rope(test_tokens, test_pos)
            CUROPE_OK = True
        except Exception as e:
            print(f'curope functional test failed: {e}')
except ImportError as e:
    print(f'curope import failed: {e}')

if CUROPE_OK:
    print('curope CUDA module: OK (fast RoPE2D available)')
else:
    print('curope CUDA module: NOT FOUND (using slow PyTorch fallback)')
    print('  To build: cd mast3r/dust3r/croco/models/curope && pip install -e . --no-build-isolation')
    print('  Or run: python build_curope.py')
    print('  This is optional but recommended for speed.')

curope import failed: cannot import name 'curope' from partially initialized module 'croco.models.curope' (most likely due to a circular import) (/root/autodl-tmp/Image-Matching-and-3D-Reconstruction/mast3r/dust3r/croco/models/curope/__init__.py)
curope CUDA module: NOT FOUND (using slow PyTorch fallback)
  To build: cd mast3r/dust3r/croco/models/curope && pip install -e . --no-build-isolation
  Or run: python build_curope.py
  This is optional but recommended for speed.


## 7. Summary

| Model | Source | Status |
|-------|--------|--------|
| MAST3R (backbone) | Official pretrained `.pth` | ✓ |
| MAST3R-ASMK (head) | Our trained checkpoint | See above |
| MAST3R-SPoC (head) | Our trained checkpoint | See above |
| DINOv2 | HuggingFace `facebook/dinov2-base` | ✓ |
| ISC | ISC21 `isc_feature_extractor` | ✓ |
| ALIKED-N16 | ALIKED torch hub | See above |
| SuperPoint | MagicLeap pretrained | See above |
| curope (RoPE2D) | CUDA extension | See above |

Next → `01_retrieval_shortlist.ipynb`